In [0]:

-- Olist Agentic Operations Control Tower
-- One-time Unity Catalog permissions setup
-- App service principal:
-- 2e629b0a-2155-438b-8e04-d02f6f48e6e7

-- 1. Catalog access
GRANT USE CATALOG ON CATALOG workspace
TO `2e629b0a-2155-438b-8e04-d02f6f48e6e7`;

-- 2. Schema access
GRANT USE SCHEMA ON SCHEMA workspace.olist_agentic_quality
TO `2e629b0a-2155-438b-8e04-d02f6f48e6e7`;

GRANT USE SCHEMA ON SCHEMA workspace.olist_governance
TO `2e629b0a-2155-438b-8e04-d02f6f48e6e7`;

GRANT USE SCHEMA ON SCHEMA workspace.olist_semantic
TO `2e629b0a-2155-438b-8e04-d02f6f48e6e7`;

-- 3. Quality data: read-only
GRANT SELECT ON TABLE workspace.olist_agentic_quality.published_run
TO `2e629b0a-2155-438b-8e04-d02f6f48e6e7`;

GRANT SELECT ON TABLE workspace.olist_agentic_quality.dq_run_summary
TO `2e629b0a-2155-438b-8e04-d02f6f48e6e7`;

GRANT SELECT ON TABLE workspace.olist_agentic_quality.dq_rule_results
TO `2e629b0a-2155-438b-8e04-d02f6f48e6e7`;

GRANT SELECT ON TABLE workspace.olist_agentic_quality.quarantine
TO `2e629b0a-2155-438b-8e04-d02f6f48e6e7`;

-- 4. Governance metadata: read-only
GRANT SELECT ON TABLE workspace.olist_governance.dq_rules
TO `2e629b0a-2155-438b-8e04-d02f6f48e6e7`;

GRANT SELECT ON TABLE workspace.olist_governance.metric_dictionary
TO `2e629b0a-2155-438b-8e04-d02f6f48e6e7`;

GRANT SELECT ON TABLE workspace.olist_governance.source_dictionary
TO `2e629b0a-2155-438b-8e04-d02f6f48e6e7`;

-- 5. Published semantic view: read-only
GRANT SELECT ON VIEW workspace.olist_semantic.v_published_orders
TO `2e629b0a-2155-438b-8e04-d02f6f48e6e7`;



SELECT
  DATE_TRUNC('MONTH', purchase_date) AS purchase_month,
  COUNT(*) AS total_orders
FROM workspace.olist_semantic.v_published_orders
WHERE purchase_date >= '2018-07-01'
GROUP BY 1
ORDER BY 1;



SELECT
    COUNT(*) AS total_rule_definitions,
    COUNT(DISTINCT rule_id) AS unique_rule_ids,
    SUM(CASE WHEN active = TRUE THEN 1 ELSE 0 END)
        AS active_rules,
    SUM(CASE WHEN active = FALSE THEN 1 ELSE 0 END)
        AS inactive_rules
FROM workspace.olist_governance.dq_rules;




SELECT
    p.run_id,
    COUNT(r.rule_id) AS evaluation_result_rows,
    COUNT(DISTINCT r.rule_id) AS unique_rules_evaluated,
    SUM(CASE WHEN r.status = 'PASSED'
        THEN 1 ELSE 0 END) AS passed_rules,
    SUM(CASE WHEN r.status = 'FAILED'
        THEN 1 ELSE 0 END) AS failed_rules,
    SUM(CASE WHEN r.status = 'ERROR'
        THEN 1 ELSE 0 END) AS error_rules,
    SUM(CASE WHEN r.status = 'SKIPPED'
        THEN 1 ELSE 0 END) AS skipped_rules
FROM workspace.olist_agentic_quality.published_run p
LEFT JOIN workspace.olist_agentic_quality.dq_rule_results r
    ON p.run_id = r.run_id
GROUP BY p.run_id;



SELECT
    r.rule_id,
    r.dataset,
    r.action,
    r.severity,
    r.records_failed,
    r.records_evaluated,
    g.business_name
FROM workspace.olist_agentic_quality.dq_rule_results r
JOIN workspace.olist_agentic_quality.published_run p
    ON r.run_id = p.run_id
LEFT JOIN workspace.olist_governance.dq_rules g
    ON r.rule_id = g.rule_id
WHERE r.status = 'FAILED'
ORDER BY r.action, r.records_failed DESC;



SELECT
    q.dataset,
    q.rule_id,
    q.action,
    COUNT(DISTINCT q.source_record_id)
        AS distinct_quarantined_records
FROM workspace.olist_agentic_quality.quarantine q
JOIN workspace.olist_agentic_quality.published_run p
    ON q.run_id = p.run_id
GROUP BY
    q.dataset,
    q.rule_id,
    q.action
ORDER BY q.dataset, q.rule_id;



-- 88 governance rules:
SELECT
    g.rule_id,
    g.dataset,
    g.business_name,
    g.severity,
    g.action,
    r.status,
    r.records_failed,
    g.failure_condition
FROM workspace.olist_governance.dq_rules g
CROSS JOIN workspace.olist_agentic_quality.published_run p
LEFT JOIN workspace.olist_agentic_quality.dq_rule_results r
    ON g.rule_id = r.rule_id
    AND p.run_id = r.run_id
WHERE g.active = TRUE
ORDER BY
    CASE WHEN r.status = 'FAILED' THEN 0 ELSE 1 END,
    g.dataset,
    g.rule_id;


-- why a particular action was taken:
SELECT
    rule_id,
    business_name,
    plain_english_description,
    business_reason,
    failure_condition,
    why_this_action,
    downstream_effect
FROM workspace.olist_governance.dq_rules
WHERE rule_id = 'orders.delivered_time';


-- failed rules:
WITH active_run AS (
    SELECT run_id
    FROM workspace.olist_agentic_quality.published_run
),

rule_results AS (
    SELECT r.*
    FROM workspace.olist_agentic_quality.dq_rule_results r
    JOIN active_run p ON r.run_id = p.run_id
),

quality_summary AS (
    SELECT
        COUNT(*) AS evaluated_rules,
        SUM(CASE WHEN status = 'PASSED' THEN 1 ELSE 0 END) AS passed_rules,
        SUM(CASE WHEN status = 'FAILED' THEN 1 ELSE 0 END) AS failed_rules
    FROM rule_results
),

quarantine_summary AS (
    SELECT
        COALESCE(SUM(distinct_rows), 0) AS quarantined_rows
    FROM (
        SELECT
            q.dataset,
            COUNT(DISTINCT q.source_record_id) AS distinct_rows
        FROM workspace.olist_agentic_quality.quarantine q
        JOIN active_run p ON q.run_id = p.run_id
        GROUP BY q.dataset
    )
)

SELECT
    r.rule_id,
    r.dataset,
    g.business_name,
    r.action,
    r.severity,
    r.records_failed AS failed_evaluations,
    g.plain_english_description,
    g.downstream_effect,
    s.evaluated_rules,
    s.passed_rules,
    s.failed_rules,
    q.quarantined_rows
FROM rule_results r
LEFT JOIN workspace.olist_governance.dq_rules g
    ON r.rule_id = g.rule_id
CROSS JOIN quality_summary s
CROSS JOIN quarantine_summary q
WHERE r.status = 'FAILED'
ORDER BY r.records_failed DESC;